# Published results

> **Placeholder:** No canonical timings have been published. Smoke output is diagnostic only.

# Full-workflow optimization and Pareto scaling

This authoritative notebook replaces both standalone optimization and Pareto notebooks. `n_zones=1` is the canonical optimization baseline; both studies cover exactly `[1, 10, 50, 100]`.

## Methodology and environment

Select `smoke` for structural checks or `full` for intentional measurement. Standard scaling uses the full-workflow SciPy SLSQP+AD problem on CPU and CUDA: packaged electricity price, one shared broadcast valve schedule, summed zone costs, and every zone's heating/cooling limits. Full mode uses the example-derived 300-iteration budget and five repetitions; convergence, comfort-limit violation, objective quality, raw timings, median/spread, and speedup eligibility remain explicit.

Pareto scaling uses the batched model layout and functional execution. CUDA uses `execution_backend="cuda_graph"`; CPU fallback uses `execution_backend="eager"`. It compares SLSQP direct shooting with the IPOPT solver using collocation transcription and `hessian="exact"`; IPOPT receives the exact sparse segment-local Lagrangian Hessian. The Pareto sweep uses `batched_prepass=True` to generate parallel warm starts. Batched same-class components remain batched inside the functional one-step map. Comfort terms are soft penalties, while the epsilon row is the only non-dynamics hard inequality. Preflight records controls, boundary states, dynamics rows, the epsilon row, and sparse Jacobian/Hessian nonzeros. Both studies checkpoint every valid raw row.

In [ ]:
# Colab bootstrap: configure a branch, tag, or immutable commit SHA.
import os
import pathlib
import subprocess
import sys


GIT_REF = "main"
REPOSITORY = "https://github.com/JBjoernskov/Twin4Build.git"
if "google.colab" in sys.modules:
    root = pathlib.Path("/content/Twin4Build")
    if not root.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(root)], check=True)
    subprocess.run(["git", "-C", str(root), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(root), "checkout", GIT_REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(root)], check=True)
else:
    root = pathlib.Path.cwd()
    if root.name == "benchmarks":
        root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from benchmarks.common import (
    BenchmarkConfig,
    OPTIMIZATION_MATRIX,
    PARETO_SOLVERS,
    run_optimization_scaling,
    ZONE_COUNTS,
    environment_metadata,
    seed_everything,
    serialize_results,
)

config = BenchmarkConfig(mode=os.environ.get("T4B_BENCHMARK_MODE", "smoke"))
seed_everything(config.seed)
assert ZONE_COUNTS == [1, 10, 50, 100]
environment_metadata(GIT_REF)

In [ ]:
print("Zone counts:", ZONE_COUNTS)
print("Standard optimization matrix:", OPTIMIZATION_MATRIX)
print("Pareto solvers:", PARETO_SOLVERS)
print("Pareto device: CUDA preferred; eager CPU fallback only when unavailable")
rows = run_optimization_scaling(config)
rows

In [ ]:
result_path = serialize_results("optimization_scaling", config, rows, GIT_REF)
print(result_path)

## Interpretation

For standard optimization, publish CPU/CUDA comparisons only for converged rows that satisfy the reported comfort limits. For Pareto scaling, compare SLSQP direct shooting and IPOPT collocation only when endpoint consistency, epsilon feasibility, continuity defect, rollout/readout parity, front coverage, and point count are valid. IPOPT rows report the exact sparse collocation Lagrangian Hessian. Model layout, execution mode, execution backend, batching mapping, fixed budgets, safeguards, and device fallback are part of the experiment.